Model training and Eval

Steps:
  1. Load cycle-level features
  2. Run LOEO CV with a Ridge baseline
  3. Run LOEO CV with Random Forest
  4. Fit final RF on all dev units
  5. Predict on test set; report overall and per-unit metrics
  6. Plot predicted vs true RUL per test unit
  7. Plot Random Forest feature importances

In [ ]:
import sys
sys.path.append("..")

from src.logging_config import setup_logging
setup_logging()

import logging
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.features import load_parquet
from src.models import (
    leave_one_engine_out_cv, fit_final,
    ridge_factory, rf_factory, feature_cols
)
from src.evaluation import evaluate, per_unit_evaluation

logger = logging.getLogger("phase4")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

Load Features

In [5]:
dev = load_parquet("dev_cycle_features")
test = load_parquet("test_cycle_features")
logger.info("Loaded dev=%s, test=%s", dev.shape, test.shape)
logger.info("Dev units: %s", sorted(dev["unit"].unique()))
logger.info("Test units: %s", sorted(test["unit"].unique()))

13:06:40 | INFO    | src.features | Reading /Users/alan/Desktop/ML_Final_Project/data/interim/dev_cycle_features.parquet
13:06:40 | INFO    | src.features | Reading /Users/alan/Desktop/ML_Final_Project/data/interim/test_cycle_features.parquet
13:06:40 | INFO    | phase4 | Loaded dev=(446, 153), test=(202, 153)
13:06:40 | INFO    | phase4 | Dev units: [np.int32(2), np.int32(5), np.int32(10), np.int32(16), np.int32(18), np.int32(20)]
13:06:40 | INFO    | phase4 | Test units: [np.int32(11), np.int32(14), np.int32(15)]


Ridge baseline

In [6]:
logger.info("Ridge baseline: leave-one-engine-out CV")
ridge_cv = leave_one_engine_out_cv(
    dev, target="RUL_pw",
    model_factory=ridge_factory(alpha=1.0),
    use_scaler=True,
)
logger.info("Ridge CV results:\n%s", ridge_cv)
logger.info("Ridge mean RMSE: %.3f, mean NASA: %.1f", ridge_cv["rmse"].mean(), ridge_cv["nasa_score"].mean())

13:06:43 | INFO    | phase4 | Ridge baseline: leave-one-engine-out CV
13:06:43 | INFO    | src.models | LOEO CV: 148 features, target=RUL_pw
13:06:43 | INFO    | src.models |   holdout=Unit 2: RMSE=6.740, NASA=133.1
13:06:43 | INFO    | src.models |   holdout=Unit 5: RMSE=6.720, NASA=147.2
13:06:43 | INFO    | src.models |   holdout=Unit 10: RMSE=6.196, NASA=132.5
13:06:43 | INFO    | src.models |   holdout=Unit 16: RMSE=7.839, NASA=129.1
13:06:43 | INFO    | src.models |   holdout=Unit 18: RMSE=6.654, NASA=122.7
13:06:43 | INFO    | src.models |   holdout=Unit 20: RMSE=7.722, NASA=134.2
13:06:43 | INFO    | phase4 | Ridge CV results:
               n_train  n_val      rmse  nasa_score
held_out_unit                                      
2                  371     75  6.739677  133.060491
5                  357     89  6.720010  147.181115
10                 364     82  6.196189  132.509740
16                 383     63  7.838792  129.137538
18                 375     71  6.653849  122.

Random Forest

In [7]:
logger.info("Random Forest: leave-one-engine-out CV")
rf_cv = leave_one_engine_out_cv(
    dev, target="RUL_pw",
    model_factory=rf_factory(n_estimators=300),
    use_scaler=False,
)
logger.info("RF CV results:\n%s", rf_cv)
logger.info("RF mean RMSE: %.3f, mean NASA: %.1f", rf_cv["rmse"].mean(), rf_cv["nasa_score"].mean())

logger.info("CV summary: Ridge vs Random Forest")
cv_summary = pd.DataFrame({
    "ridge_rmse": ridge_cv["rmse"],
    "rf_rmse": rf_cv["rmse"],
    "ridge_nasa": ridge_cv["nasa_score"],
    "rf_nasa": rf_cv["nasa_score"],
})
logger.info("\n%s", cv_summary.round(2))

13:09:02 | INFO    | phase4 | Random Forest: leave-one-engine-out CV
13:09:02 | INFO    | src.models | LOEO CV: 148 features, target=RUL_pw
13:09:02 | INFO    | src.models |   holdout=Unit 2: RMSE=8.243, NASA=153.8
13:09:02 | INFO    | src.models |   holdout=Unit 5: RMSE=8.721, NASA=175.2
13:09:03 | INFO    | src.models |   holdout=Unit 10: RMSE=8.730, NASA=171.7
13:09:03 | INFO    | src.models |   holdout=Unit 16: RMSE=11.074, NASA=183.9
13:09:03 | INFO    | src.models |   holdout=Unit 18: RMSE=9.625, NASA=165.5
13:09:03 | INFO    | src.models |   holdout=Unit 20: RMSE=9.751, NASA=167.3
13:09:03 | INFO    | phase4 | RF CV results:
               n_train  n_val       rmse  nasa_score
held_out_unit                                       
2                  371     75   8.242657  153.755200
5                  357     89   8.721462  175.217980
10                 364     82   8.729958  171.675936
16                 383     63  11.074343  183.876738
18                 375     71   9.625010  

Train final Ridge on all dev units


In [8]:
logger.info("Training final Ridge on all dev units")
final_model = fit_final(
    dev, target="RUL_pw",
    model_factory=ridge_factory(alpha=1.0),
    use_scaler=True,
)

13:09:11 | INFO    | phase4 | Training final Ridge on all dev units
13:09:11 | INFO    | src.models | Final model trained on 446 cycles, 148 features


Predict on test, run eval

In [9]:
test_pred = final_model.predict(test)
test_eval = test.assign(y_pred=test_pred)

logger.info("Test set evaluation")
evaluate(test_eval["RUL_pw"].values, test_eval["y_pred"].values, label="test (overall)")

per_unit = per_unit_evaluation(test_eval, y_true_col="RUL_pw", y_pred_col="y_pred")
logger.info("Per-test-unit metrics:\n%s", per_unit.round(2))



13:09:12 | INFO    | phase4 | Test set evaluation
13:09:12 | INFO    | src.evaluation | [test (overall)] RMSE=8.085, NASA=419.0, n=202
13:09:12 | INFO    | phase4 | Per-test-unit metrics:
      n_cycles  rmse  nasa_score
unit                            
11          59  7.98      122.96
14          76  8.58      165.73
15          67  7.58      130.30


Predicted vs true RUL plot per test unit

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, unit in zip(axes, sorted(test_eval["unit"].unique())):
    g = test_eval[test_eval["unit"] == unit].sort_values("cycle")
    ax.plot(g["cycle"], g["RUL_pw"], color="#222", linewidth=1.8,
            label="True (clipped)")
    ax.plot(g["cycle"], g["y_pred"], color="#2c5f8d", linewidth=1.4,
            alpha=0.85, label="Predicted")
    ax.set_title(f"Unit {unit}")
    ax.set_xlabel("Flight cycle")
    ax.set_ylabel("RUL (clipped at 50)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("Ridge predictions on test units", fontsize=12)
fig.tight_layout()
pred_path = FIG_DIR / "test_predictions_per_unit.png"
fig.savefig(pred_path, dpi=150, bbox_inches="tight")
plt.close(fig)
logger.info("Saved %s", pred_path)

12:57:48 | INFO    | phase4 | Saved ../reports/figures/test_predictions_per_unit.png


Feature importance 

In [14]:
ridge_coefs = pd.Series(
    final_model.model.coef_,
    index=final_model.features,
).abs().sort_values(ascending=False)

top_n = 20
logger.info("Top %d Ridge coefficients (|coef|, standardized features):\n%s",
            top_n, ridge_coefs.head(top_n).round(4))

fig, ax = plt.subplots(figsize=(8, 6))
ridge_coefs.head(top_n).iloc[::-1].plot(kind="barh", ax=ax, color="#2c3e50")
ax.set_xlabel("|coefficient| (standardized features)")
ax.set_title(f"Top {top_n} feature importances — Ridge regression")
fig.tight_layout()
imp_path = FIG_DIR / "feature_importances.png"
fig.savefig(imp_path, dpi=150, bbox_inches="tight")
plt.close(fig)
logger.info("Saved %s", imp_path)

12:58:05 | INFO    | phase4 | Top 20 Ridge coefficients (|coef|, standardized features):
T50_max     7.4032
T50_mean    5.1985
P50_std     4.8220
T50_r1      4.4866
T50_std     4.2639
T50_r2      4.1566
Wf_r0       3.9862
T2_min      3.9358
T50_r4      3.5381
T24_min     3.4913
Nf_r0       3.2379
T50_r0      3.2133
T30_r0      3.2015
Wf_std      3.0768
T48_max     2.9452
P50_r3      2.9416
Nf_r2       2.9120
Nf_min      2.8151
Nf_std      2.8036
T2_max      2.7918
dtype: float64
12:58:05 | INFO    | phase4 | Saved ../reports/figures/feature_importances.png


Save predictions for the report


In [15]:
test_eval[["unit", "cycle", "RUL", "RUL_pw", "y_pred"]].to_csv(
    "../data/interim/test_predictions.csv", index=False,
)
logger.info("Saved test_predictions.csv")

12:58:17 | INFO    | phase4 | Saved test_predictions.csv
